In [7]:
import numpy as np
import matplotlib.pyplot as plt
from functools import lru_cache
from scipy.optimize import brentq
import mpmath

In [8]:
 def Z(N,beta,eps_mp,):
    """Canonical partition function for N fermions at global beta."""
    if N == 0:  return mpmath.mpf(1)
    if N < 0:   return mpmath.mpf(0)
    total = mpmath.mpf(0)
    for i in range(1,N+1):
        total += (-1)**(i+1)*Z1(i*beta,eps_mp)*Z(N-i,beta,eps_mp)
    return total/N

def c_g2(N, beta_f, r_vals, c,ks,L_f):
    mpmath.mp.dps = 180
 
    beta_mp   = mpmath.mpf(str(beta_f))
    L_mp      = mpmath.mpf(str(L_f))
    eps_np  = c * np.abs(ks)
    eps_mp    = [mpmath.mpf(str(e)) for e in eps_np]
    n_r       = len(r_vals)
 
    @lru_cache(maxsize=None)
    def Z1_mp(l):
        return sum(mpmath.exp(-l * beta_mp * e) for e in eps_mp)

 
    @lru_cache(maxsize=None)
    def Z_mp(n):
        if n == 0: return mpmath.mpf(1)
        if n < 0:  return mpmath.mpf(0)
        return sum((-1)**(i-1) * Z1_mp(i) * Z_mp(n-i)  for i in range(1, n+1)) / n
 
    ZN = Z_mp(N)
 
    phi_dd = [Z1_mp(l) / L_mp for l in range(1, N+1)]
 
    A = [sum(phi_dd[l-1] * phi_dd[m-l-1] for l in range(1, m)) for m in range(2, N+1)]
 
    def phi_l_ofd(l):
        """F_l(r_vals) = sum_{k: l*beta*eps_k < THRESHOLD} exp(-l*beta*eps_k)*exp(ikr)."""
        w_sig   = [mpmath.exp(-l * beta_mp * mpmath.mpf(str(e))) for e in eps_np]
        phase_f = np.exp(1j * ks[:, None] * r_vals[None, :])   # (n_sig, n_r) float64
        result  = [mpmath.mpc(0)] * n_r
        for ki, wk in enumerate(w_sig):
            for ri in range(n_r):
                pf = phase_f[ki, ri]
                result[ri] += wk * mpmath.mpc(str(pf.real), str(pf.imag))
        return result
 
    F_all = [phi_l_ofd(l) for l in range(1, N)]   # (N-1) lists of n_r mpmath.mpc
 
    rho2_mp = [mpmath.mpf(0)] * n_r
    for m in range(2, N+1):
        ZNm = Z_mp(N - m)
        Am  = A[m - 2]
        for ri in range(n_r):
            Bm = sum(
                F_all[l-1][ri] * F_all[m-l-1][ri].conjugate()
                for l in range(1, m)
            ) / L_mp**2
            rho2_mp[ri] += (-1)**m * ZNm * (Am - Bm)
 
    n = N / L_f
    return np.array([float((x / ZN).real) / n**2 for x in rho2_mp])

In [9]:
def gc_g2(N_target, beta_f, r_vals, c, ks):
    eps_np  = c * np.abs(ks)
    def mean_N(mu):
        return np.sum(1.0 / (np.exp(beta_f * (eps_np - mu)) + 1.0))
    mu    = brentq(lambda m: mean_N(m) - N_target,-100.0 / beta_f, 100.0 / beta_f, xtol=1e-12)
    f_k   = 1.0 / (np.exp(beta_f * (eps_np - mu)) + 1.0)
    N_avg = np.sum(f_k)
    g2 = np.array([1.0 - np.abs(np.sum(f_k * np.exp(1j * ks * r)))**2 / N_avg**2 for r in r_vals
    ])
    return mu, g2

In [10]:
L   = 10.0
c   = 1.0
N = 20
beta = 4
K_MAX = 100
n_r = 100
ns_np  = np.arange(-K_MAX, K_MAX + 1)
ks = 2.0 * np.pi * ns_np / L
xvals = np.linspace(0, L / 2, n_r)
cg220 = c_g2(20,beta,xvals,c,ks,L)
gg220 = gc_g2(20,beta,xvals,c,ks)
cg210 = c_g2(15,beta,xvals,c,ks,L)
gg210 = gc_g2(15,beta,xvals,c,ks)
cg25 = c_g2(5,beta,xvals,c,ks,L)
gg25 = gc_g2(5,beta,xvals,c,ks)

In [16]:
with plt.style.context('../include/aps.mplstyle'):
    figsize = plt.rcParams['figure.figsize']
    fig, ax = plt.subplots(constrained_layout=True, figsize=(figsize[0],figsize[1]))
    ax.plot(xvals,gg220[1],color='red', linestyle="--", label='Grand-canonical')
    ax.plot(xvals,cg220,color='red',label='N = 20')
    ax.plot(xvals,gg210[1],color='blue', linestyle="--")
    ax.plot(xvals,cg210,color='blue',label='N = 15')
    #plt.plot(xvals,gg25[1],color='green',linewidth=1.5, linestyle="--")
    #plt.plot(xvals,cg25,color='green',linewidth=2.0,label='N = 5')
    ax.axhline(1.0, color="gray", linestyle=":")
    ax.set_xlabel(r"$x$", fontsize=12)
    ax.set_ylabel(r"$g^{(2)}(x,0)$")
    ax.set_xlim(0, L / 2-0.1)
    ax.set_ylim(0.9, 1.04)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.savefig('../figures/FreeFermionPair.pdf')